In [1]:
from opt_targeted_transfers import RateTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = RateTargetedTransfers(c_bar=2.15, budget=None)
tt.fit(train_dataset=train_dataset, validation_dataset=validation_dataset)
tt.set_budget(0.5)

Fitting conditional densities vs glm spline method...


100%|██████████| 300/300 [00:11<00:00, 26.52it/s, loss=-0.0223, val_loss=-0.08]  

Final Theta: tensor([[-1.6869,  1.8743],
        [ 0.2049,  0.1105],
        [-0.1265, -0.4476],
        [-0.2721,  0.8554],
        [-0.5875, -0.0802],
        [-1.2860,  0.9281],
        [-2.1586,  2.1372]], dtype=torch.float64)


In [4]:
assignments = tt.run_opt(
   test_covariate_dataset, n_alpha=1, max_alpha=0.2, min_alpha=0.2, path="malawi_example_rate_B=0.5.csv"
)

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:11<00:00, 11.00s/it]


In [ ]:
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.35824955663870195,
 'post_transfer_poverty_rate': 0.42564770694691034,
 'policy_cost_per_capita': 0.5,
 'budget': 0.5,
 'policy_type': 'continuous_rate',
 'd': 2}

In [6]:
tt.set_budget(2.0)
tt.run_opt(
   test_covariate_dataset, n_alpha=1, max_alpha=0.2, min_alpha=0.2, path="malawi_example_rate_B=2.0.csv"
)
res = tt.evaluate(test_dataset)
res

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:10<00:00, 10.66s/it]


{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.0005330273621647837,
 'post_transfer_poverty_rate': 0.0067351698567101265,
 'policy_cost_per_capita': 2.000000000000003,
 'budget': 2.0,
 'policy_type': 'continuous_rate',
 'd': 2}

In [7]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, test_dataset=test_dataset, metrics=["post_transfer_poverty_rate",
                                                              "post_transfer_poverty_gap"], budgets=[0.05, 0.1, 0.5, 1.0, 2.0], min_alpha=0.2, max_alpha=0.2, n_alpha=1)

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.93s/it]


Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.51s/it]


Alpha range: 0.2, 0.2


...

Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.47s/it]


Alpha range: 0.2, 0.2


100%|██████████| 1/1 [00:09<00:00,  9.75s/it]


{'post_transfer_poverty_rate': {'auc': 0.5075696072966724,
  'results': [0.6113800127628596,
   0.5907430898944198,
   0.42564770694691034,
   0.21927847826252234,
   0.0067351698567101265]},
 'post_transfer_poverty_gap': {'auc': 0.41723432184716236,
  'results': [0.526732924792189,
   0.5080125505529123,
   0.35824955663870195,
   0.17104581424593882,
   0.0005330273621647837]}}